# 🤖 Franka Robot: Task & Motion Planning Tutorial

## Beginner-Friendly Guide: Language → Vision → Planning → Feasibility

This tutorial walks you through building a **complete pipeline** for a **Franka robot** that:

1. **Understands natural language** – Turns a command like *"Pick up the red mug and place it near the laptop"* into a **symbolic plan** (sequence of actions) using the **local NIM LLM** from your Docker setup.

2. **Grounds commands in the scene** – Uses an **object detection model** (YOLO) to find objects in the scene and link words like "mug" and "laptop" to actual detections.

3. **Checks feasibility** – Uses **PyBullet** with a Franka model to:
   - Compute **forward kinematics** (joint angles → end-effector pose)
   - Compute **inverse kinematics** (target pose → joint angles)
   - Check **reachability** (can the robot reach the target?)
   - Check **collisions** (would the arm or object hit something?)
   - Check **stability** (is the object supported when placed?)

By the end, you will have a pipeline that takes a natural language command and answers: **"Is this command feasible for the robot?"** with a clear yes/no and reasons.

### What You'll Learn

- **Symbolic planning**: From natural language to a list of actions (pick, place, move_to, etc.).
- **Grounding**: From abstract object names to bounding boxes and 3D positions in the scene.
- **Kinematics**: What forward and inverse kinematics are, and how to use PyBullet to compute them for the Franka.
- **Feasibility**: How to combine reachability, collision, and stability into a single feasibility check.

### Prerequisites

- Python 3.8+
- (Optional) **NVIDIA NIM** running via Docker for language parsing (see `docker-compose-nim.yml`).
- (Optional) GPU for faster YOLO and NIM; CPU is fine for learning.

Let's get started! 🚀

## 📋 Table of Contents

1. **Part 1: Setup & installation** – Install PyBullet, OpenAI client (for NIM), YOLO, and other dependencies.
2. **Part 2: Natural language → symbolic plan** – Use the local NIM LLM to parse a command into a list of robot actions.
3. **Part 3: Grounding with object detection** – Run YOLO on the scene image and match object names to detections.
4. **Part 4: PyBullet & Franka** – Load the Franka in PyBullet, compute FK/IK, and introduce collision/reachability.
5. **Part 5: Feasibility checking** – Implement reachability, collision, and stability checks.
6. **Part 6: Full pipeline** – Tie everything together: command → plan → grounding → feasibility.

---
# Part 1: Setup & Prerequisites

We need these packages:

- **pybullet** – Physics simulation and robot kinematics (Franka model, FK/IK, collision).
- **openai** – Python client that talks to NIM’s OpenAI-compatible API (for the LLM).
- **pydantic** – To define and validate the symbolic action structure (e.g. action type, target, location).
- **ultralytics** – YOLO object detection for grounding.
- **opencv-python**, **numpy**, **Pillow** – Image loading and processing for the camera/scene.

Run the cell below to install them. If you already have them, you can skip or re-run without issues.

In [1]:
!pwd

/dli/task/Complete_Robot_Tutorial


In [2]:
# Install dependencies for the tutorial
# Run this cell once. -q reduces output.

!pip install pybullet -q
!pip install openai pydantic python-dotenv requests -q
!pip install ultralytics opencv-python numpy pillow -q

print("✅ Installation complete.")
print("   - pybullet: robot simulation and FK/IK")
print("   - openai: NIM LLM client")
print("   - pydantic: action validation")
print("   - ultralytics: YOLO object detection")

✅ Installation complete.
   - pybullet: robot simulation and FK/IK
   - openai: NIM LLM client
   - pydantic: action validation
   - ultralytics: YOLO object detection


In [3]:
# Import libraries used throughout the tutorial
# We group them by purpose so you can see what each part of the pipeline uses.

# --- Language & validation ---
from openai import OpenAI
from pydantic import BaseModel, Field, model_validator
from typing import List, Optional, Dict, Tuple, Any
from enum import Enum
import json
import os

# --- Vision & grounding ---
import cv2
import numpy as np
from PIL import Image

# --- Physics & robot (PyBullet) ---
import pybullet as p
import pybullet_data

# --- Utilities ---
from dataclasses import dataclass
import warnings
warnings.filterwarnings("ignore")

print("✅ All imports successful.")

✅ All imports successful.


pybullet build time: Feb 12 2026 01:34:48


---
# Part 2: Natural Language → Symbolic Plan (using NIM LLM)

## What we do here

The user gives a **natural language command** (e.g. *"Pick up the red mug and place it near the laptop"*). We send it to the **local NIM LLM** (running in Docker) and ask it to output a **symbolic plan**: a list of structured actions that our robot can execute.

Each action has:
- **action**: one of `pick`, `place`, `move_to`, `grasp`, `release`, etc.
- **target**: the object to act on (e.g. `"red mug"`, `"mug"`).
- **location** (optional): for place/move_to, where to put or go (e.g. `"laptop"`).
- **relation** (optional): spatial relation like `"near"`, `"on"`, `"under"`.

NIM exposes an **OpenAI-compatible API** on port **8000**. We use the `openai` Python client with `base_url="http://localhost:8000/v1"` (or `http://nim:8000/v1` if you run the notebook inside Docker). The LLM is prompted to return valid JSON that we then parse into our `RobotAction` objects.

In [4]:
# ============================================================
# Step 2.1: Define the action types and the RobotAction schema
# ============================================================
# This is the "grammar" of what the robot can do. The LLM must output
# only these action types so we can validate and execute them.

class ActionType(str, Enum):
    """Allowed robot action types. The LLM should map natural language to these."""
    PICK = "pick"
    PLACE = "place"
    MOVE_TO = "move_to"
    GRASP = "grasp"
    RELEASE = "release"
    OPEN = "open"
    CLOSE = "close"
    PUSH = "push"
    PULL = "pull"


class RobotAction(BaseModel):
    """
    One symbolic action in the plan.
    - action: what to do (pick, place, ...)
    - target: object to act on (e.g. "mug", "red mug")
    - location: for place/move_to, the reference object or place
    - relation: e.g. "near", "on", "under"
    """
    action: ActionType = Field(description="The type of action")
    target: str = Field(description="The object or target of the action", min_length=1)
    location: Optional[str] = Field(None, description="Location or reference object")
    relation: Optional[str] = Field(None, description="Spatial relation: near, on, under, etc.")

    @model_validator(mode="after")
    def validate_location_for_action(self):
        """Place and move_to must have a location."""
        if self.action in (ActionType.PLACE, ActionType.MOVE_TO) and not self.location:
            raise ValueError(f"{self.action.value} requires a location")
        return self

    def to_tuple(self) -> Tuple[str, str, Optional[str], Optional[str]]:
        """Return (action, target, location, relation) for easy use."""
        return (self.action.value, self.target, self.location, self.relation)

print("✅ Action types and RobotAction schema defined.")

✅ Action types and RobotAction schema defined.


In [5]:
# ============================================================
# Step 2.2: NIM LLM client and prompt for symbolic plan
# ============================================================
# NIM runs in Docker and exposes OpenAI-compatible API on port 8000.
# Use base_url="http://localhost:8000/v1" when running on host;
# use base_url="http://nim:8000/v1" when notebook runs inside Docker.

# Try both common hostnames so it works in different environments
NIM_BASE_URL = os.environ.get("NIM_BASE_URL", "http://nim:8000/v1")
# If you run Jupyter inside Docker next to NIM, uncomment:
NIM_BASE_URL = "http://nim:8000/v1"
# Model ID your NIM exposes (run the "List NIM models" cell below to see available IDs)
NIM_MODEL = os.environ.get("NIM_MODEL", "nvidia/llama-3.3-nemotron-super-49b-v1.5")
print(NIM_MODEL)
def get_nim_client() -> Optional[OpenAI]:
    """Create OpenAI client pointing to NIM. Returns a client; use_llm will fail at first call if NIM is down."""
    try:
        client = OpenAI(base_url=NIM_BASE_URL, api_key="not-needed")
        return client
    except Exception as e:
        print("⚠️ NIM client creation failed: %s" % e)
        return None

# Optional: check NIM at startup
_nim = get_nim_client()
if _nim:
    print("✅ NIM LLM client ready (base_url=%s)" % NIM_BASE_URL)
else:
    print("⚠️ NIM not connected. We'll use a fallback parser for demo.")

nvidia/llama-3.3-nemotron-super-49b-v1.5
✅ NIM LLM client ready (base_url=http://nim:8000/v1)


In [6]:
# List models NIM exposes (run this to get the right value for NIM_MODEL)
# Use the "id" shown below in the NIM client cell: NIM_MODEL = "that-id"
client = get_nim_client()
if client:
    try:
        models = client.models.list()
        print("Models available on NIM:")
        for m in models.data:
            print("  id: %s" % m.id)
        if models.data:
            print("\nTo use the first one, set in the NIM client cell: NIM_MODEL = \"%s\"" % models.data[0].id)
    except Exception as e:
        print("Could not list models: %s" % e)
else:
    print("NIM client not available. Run the NIM client cell first.")

Models available on NIM:
  id: nvidia/llama-3.3-nemotron-super-49b-v1.5

To use the first one, set in the NIM client cell: NIM_MODEL = "nvidia/llama-3.3-nemotron-super-49b-v1.5"


In [ ]:
# ============================================================
# Step 2.3: Parse natural language command into symbolic plan via NIM
# ============================================================
# We send a system + user prompt so the LLM returns a JSON array of actions.
# Each action: {"action": "pick"|"place"|..., "target": "...", "location": "..." (optional), "relation": "..." (optional)}

SYSTEM_PROMPT = """You are a robot command parser. Given a natural language command, output ONLY a JSON array of robot actions. No <think> tags, no explanation, no other text.
Each action: "action" (pick, place, move_to, grasp, release, open, close, push, or pull), "target" (string). For place/move_to add "location" (string) and optionally "relation" (e.g. near, on, under).
Reply with nothing but the JSON array. Example:
Command: "Pick up the mug and place it near the laptop"
Output: [{"action":"pick","target":"mug"},{"action":"place","target":"mug","location":"laptop","relation":"near"}]"""


def _extract_json_array(text: str) -> str:
    """Strip markdown, <think> blocks, and extract a JSON array from LLM output."""
    if not text or not text.strip():
        raise ValueError("LLM returned empty content")
    text = text.strip()
    # Remove <think>...</think> blocks (Nemotron and some models output chain-of-thought here)
    if "</think>" in text:
        # Keep only the part after the last </think> (the actual answer is usually there)
        text = text.split("</think>")[-1].strip()
    if "<think>" in text:
        # If there's still a <think> without closing, drop everything before the last </think> already done
        text = text.split("<think>")[-1].strip()
    # Remove markdown code fence (e.g. ```json ... ``` or ``` ... ```)
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines[1:] if l.strip() != "```"]
        text = "\n".join(lines)
    # Find first '[' and last ']' to get the JSON array (handles extra text before/after)
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON array in response. Got: %s" % repr(text)[:300])
    text = text[start : end + 1]
    return text.strip()


def _call_nim_llm(client: OpenAI, command: str, model: str) -> List[RobotAction]:
    """Call NIM chat completion and parse response into list of RobotAction."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Command: {command}"},
        ],
        max_tokens=512,
        temperature=0.1,
    )
    raw = response.choices[0].message.content
    if raw is None:
        raw = ""
    text = _extract_json_array(raw)
    try:
        actions_dict = json.loads(text)
    except json.JSONDecodeError as je:
        raise ValueError("Invalid JSON from NIM (raw snippet): %s" % repr(raw[:400])) from je
    return [RobotAction(**a) for a in actions_dict]


def parse_command_with_nim(command: str, client: Optional[OpenAI] = None) -> List[RobotAction]:
    """
    Use NIM LLM to convert natural language command into a list of RobotAction.
    Uses NIM_MODEL; on 404 (model not found) tries the first model NIM exposes.
    """
    if client is None:
        client = get_nim_client()

    if not client:
        return _parse_command_fallback(command)

    model = NIM_MODEL
    try:
        return _call_nim_llm(client, command, model)
    except Exception as e:
        err_str = str(e)
        if "404" in err_str or "does not exist" in err_str.lower():
            try:
                models = client.models.list()
                if models.data:
                    model = models.data[0].id
                    return _call_nim_llm(client, command, model)
            except Exception:
                pass
        # Show error (for JSON failures, the message may include a snippet of what NIM returned)
        print("⚠️ LLM parse failed (%s), using fallback." % e)

    return _parse_command_fallback(command)


# def _parse_command_fallback(command: str) -> List[RobotAction]:
#     """
#     Simple fallback when NIM is not available. Handles pick, place, move_to patterns.
#     """
#     cmd = command.lower().strip()
#     # Normalize "moveit" (no space) so "move it to" is matched
#     cmd = cmd.replace("moveit", "move it")
#     actions = []

#     # "pick X and ..." or "pick X"
#     if "pick" in cmd or "grab" in cmd or "take" in cmd:
#         for verb in ["pick up the", "pick the", "grab the", "take the", "pick up a", "pick a"]:
#             if verb in cmd:
#                 rest = cmd.split(verb, 1)[-1].strip()
#                 target = rest.split()[0] if rest else "object"
#                 if target in ("it", "that"):
#                     target = "object"
#                 actions.append(RobotAction(action=ActionType.PICK, target=target))
#                 break

#     # "place it near/on Y" or "put it ..."
#     if "place" in cmd or "put" in cmd:
#         for phrase in ["place it near", "put it near", "place it on", "put it on", "place the", "put the"]:
#             if phrase in cmd:
#                 rest = cmd.split(phrase, 1)[-1].strip()
#                 parts = rest.split()
#                 if len(parts) >= 2:
#                     relation, location = parts[0], parts[1]
#                 else:
#                     relation, location = "near", parts[0] if parts else "table"
#                 target = actions[0].target if actions else "object"
#                 actions.append(RobotAction(action=ActionType.PLACE, target=target, location=location, relation=relation))
#                 break

#     # "move it to X" or "move to X" (with or without prior pick/place)
#     for phrase in ["move it to the", "move it to", "move to the", "move to"]:
#         if phrase in cmd:
#             rest = cmd.split(phrase, 1)[-1].strip()
#             location = rest.split()[0] if rest else "table"
#             target = actions[0].target if actions else "object"
#             actions.append(RobotAction(action=ActionType.MOVE_TO, target=target, location=location))
#             break

#     if not actions and "move" in cmd:
#         rest = cmd.replace("move to the", "").replace("move to", "").strip()
#         target = rest.split()[0] if rest else "table"
#         actions.append(RobotAction(action=ActionType.MOVE_TO, target=target, location=target))

#     if not actions:
#         raise ValueError("Could not parse command: %s" % command)
#     return actions


# --- Test ---
test_cmd = "Pick up the red_mug and moveit to the kitchen"
try:
    plan = parse_command_with_nim(test_cmd)
    print("Parsed plan for: '%s'" % test_cmd)
    for i, a in enumerate(plan):
        print("  %d. %s" % (i + 1, a.to_tuple()))
except Exception as e:
    print("Parse error:", e)

⚠️ LLM parse failed (No JSON array in response. Got: 'Okay, let\'s tackle this command. The user said, "Pick up the red_mug and moveit to the kitchen." First, I need to parse the actions correctly.\n\nThe first part is "Pick up the red_mug." That\'s straightforward. The action here is "pick" with the target "red_mug." So that\'s the first element in t), using fallback.
Parsed plan for: 'Pick up the red_mug and moveit to the kitchen'
  1. ('pick', 'red_mug', None, None)
  2. ('move_to', 'red_mug', 'kitchen', None)


---
# Part 3: Grounding – Object Detection in the Scene

## What we do here

The **symbolic plan** uses abstract names like "mug" and "laptop". **Grounding** means linking those names to **real objects in the scene**: we run an **object detection model** (YOLO) on the scene image to get bounding boxes and labels, then match each plan symbol to one or more detections.

We use **Ultralytics YOLO** (e.g. YOLOv8). You can use the provided `yolov8n.pt` model or let the library download it. Detections give us:
- **label**: class name (e.g. "cup", "laptop", "book")
- **bbox**: [x, y, width, height] in image coordinates
- **confidence**: score in [0, 1]
- **center**: (cx, cy) for easy distance/reachability later

We also convert image coordinates to a **simple 3D world frame** (e.g. table at z=0, x/y from pixel position) so the same positions can be used in PyBullet for reachability and collision checks.

In [8]:
# ============================================================
# Step 3.1: DetectedObject and running YOLO on the scene
# ============================================================

@dataclass
class DetectedObject:
    """
    One detected object: label, 2D bbox in image, and optional 3D position
    for use in PyBullet (e.g. on table plane).
    """
    label: str
    bbox: List[float]   # [x, y, width, height]
    confidence: float
    center_2d: Tuple[float, float]  # (cx, cy) in image
    # 3D position in world (meters), for robot planning. Set by grounding.
    position_3d: Optional[Tuple[float, float, float]] = None

    def distance_to_2d(self, other: "DetectedObject") -> float:
        """Euclidean distance between centers in image."""
        dx = self.center_2d[0] - other.center_2d[0]
        dy = self.center_2d[1] - other.center_2d[1]
        return np.sqrt(dx * dx + dy * dy)


def run_object_detection(image_path: str, model_path: Optional[str] = None) -> List[DetectedObject]:
    """
    Run YOLO on the image and return a list of DetectedObject.
    Uses yolov8n.pt in the project folder if present, else default YOLOv8n.
    """
    from ultralytics import YOLO

    # Use local yolov8n.pt in project folder if present
    if model_path is None:
        for candidate in ["yolov8n.pt", os.path.join(os.getcwd(), "yolov8n.pt")]:
            if os.path.isfile(candidate):
                model_path = candidate
                break
        else:
            model_path = "yolov8n.pt"  # Ultralytics will download

    model = YOLO(model_path)
    results = model(image_path, verbose=False)[0]

    detections = []
    if results.boxes is None:
        return detections

    names = results.names or {}
    for box in results.boxes:
        cls_id = int(box.cls[0])
        label = names.get(cls_id, "object")
        xyxy = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        x1, y1, x2, y2 = xyxy
        w, h = x2 - x1, y2 - y1
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        detections.append(DetectedObject(
            label=label,
            bbox=[float(x1), float(y1), float(w), float(h)],
            confidence=conf,
            center_2d=(float(cx), float(cy)),
        ))
    return detections

In [9]:
# ============================================================
# Step 3.2: Assign 3D positions and ground plan to detections
# ============================================================
# We assume a fixed camera view: image coordinates map to a table plane.
# Simple mapping: image (0,0) -> world (x_min, y_min), (width, height) -> (x_max, y_max), z = table_height.
# You can replace this with real camera intrinsics + table plane for a real robot.

# Table and workspace in meters (Franka typically works in ~0.8 m reach)
TABLE_Z = 0.0
#if the image is not 640x480, you should change the image width and height
#This is to accomodate the YOLO model that often runs at 640x480
IMAGE_WIDTH = 640
IMAGE_HEIGHT = 480
#Why ±0.4 m: Franka’s reach is about 0.8 m; a 0.8 m wide strip (from -0.4 to +0.4) is a safe, symmetric workspace in front of the robot. So “left” in the image is roughly x = -0.4, “right” is x = +0.4.
WORLD_X_MIN, WORLD_X_MAX = -0.4, 0.4
#Y is forward for the robot . Why ±0.3 m: A 0.6 m deep table is typical for a desk; it fits inside reach and keeps the math simple. So “bottom” of the image (closer to camera/robot) maps to one Y edge, “top” to the other.
WORLD_Y_MIN, WORLD_Y_MAX = -0.3, 0.3


def pixel_to_world(cx: float, cy: float, z: float = TABLE_Z) -> Tuple[float, float, float]:
    """Convert image center (cx, cy) to 3D world (x, y, z) on table plane."""
    """in images, y usually increases downward. In many robot frames, Y increases “forward” (away from the robot / toward the table).
    So “top of image” (small cy) should often be “further” (e.g. +0.3) and “bottom of image” (large cy) “closer” (e.g. -0.3). That’s why we subtract the fraction from WORLD_Y_MAX:
    cy = 0 (top) → y = WORLD_Y_MAX (+0.3)
    cy = HEIGHT (bottom) → y = WORLD_Y_MIN (-0.3)
    So image Y is mapped linearly to world Y, but inverted so that image “up” corresponds to world “far” and image “down” to world “near."""
    # Convert image coordinates to world coordinates
    """cx / IMAGE_WIDTH is a fraction in [0, 1]: 0 = left, 1 = right.
    Multiply by the world width (WORLD_X_MAX - WORLD_X_MIN) = 0.8 m → you get a length in meters from the left world edge.
    Add WORLD_X_MIN (-0.4) so that:
    cx = 0 → x = -0.4
    cx = IMAGE_WIDTH → x = +0.4
    So image X is mapped linearly to world X"""
    x = WORLD_X_MIN + (cx / IMAGE_WIDTH) * (WORLD_X_MAX - WORLD_X_MIN)
    y = WORLD_Y_MAX - (cy / IMAGE_HEIGHT) * (WORLD_Y_MAX - WORLD_Y_MIN)  # y flipped
    return (x, y, z)


def ground_detections_to_world(detections: List[DetectedObject]) -> None:
    """Fill position_3d for each detection using pixel_to_world."""
    for d in detections:
        d.position_3d = pixel_to_world(d.center_2d[0], d.center_2d[1], TABLE_Z)


def match_symbol_to_detection(symbol: str, detections: List[DetectedObject]) -> Optional[DetectedObject]:
    """
    Match a plan symbol (e.g. 'mug', 'red mug') to the best detection.
    Uses simple string matching: symbol in label or label in symbol.
    Returns the highest-confidence matching detection, or None.
    """
    symbol_lower = symbol.lower().strip()
    best = None
    best_score = -1.0
    for d in detections:
        label_lower = d.label.lower()
        # Match if symbol is contained in label or label in symbol (e.g. 'mug' <-> 'cup')
        if symbol_lower in label_lower or label_lower in symbol_lower:
            if d.confidence > best_score:
                best_score = d.confidence
                best = d
    return best


# Test: run detection on a sample scene if available
_sample = "scene_1.jpg"
if os.path.isfile(_sample):
    dets = run_object_detection(_sample)
    ground_detections_to_world(dets)
    print("Detected %d objects in %s" % (len(dets), _sample))
    for d in dets[:5]:
        print("  ", d.label, d.confidence, "->", d.position_3d)
else:
    print("No scene_1.jpg found; run detection on your own image path later.")

Detected 0 objects in scene_1.jpg


---
# Part 4: PyBullet & Franka – Kinematics and Collision

## What we do here

We use **PyBullet** to simulate the **Franka Emika Panda** arm and the table/objects. This lets us:

1. **Load the robot** from a URDF and set initial joint angles.
2. **Forward kinematics (FK)**: Given joint angles, compute the end-effector pose (position + orientation) in the world frame.
3. **Inverse kinematics (IK)**: Given a target pose for the end-effector, find joint angles that achieve it (if any).
4. **Reachability**: Check if a target 3D point is within the robot’s workspace by testing whether IK finds a solution.
5. **Collision checking**: Check if the robot (at a given configuration) collides with itself or with obstacles (table, objects).
6. **Stability** (simple): For place actions, check that the target location is on a supported surface (e.g. table).

PyBullet provides:
- `p.getLinkState(robot_id, link_index)` for FK (link poses).
- `p.calculateInverseKinematics(...)` for IK.
- `p.getClosestPoints(bodyA, bodyB, ...)` or collision queries for collision detection.

We use the **Franka Panda** URDF. PyBullet’s `pybullet_data` may include a panda model; if not, we use a built-in or a documented path to download one.

In [10]:
# ============================================================
# Step 4.1: Start PyBullet and load Franka Panda (or fallback)
# ============================================================
# We use DIRECT mode (no GUI) so the notebook runs quickly. Use GUI for debugging.

def start_pybullet_environment(use_gui: bool = False):
    """Start PyBullet and add search path for robot URDFs."""
    if use_gui:
        client = p.connect(p.GUI)
    else:
        client = p.connect(p.DIRECT)
    p.setGravity(0, 0, -9.81)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    return client


def load_franka_panda():
    """
    Load Franka Panda arm. Tries pybullet_data franka_panda folder.
    If not found, loads a simple box as placeholder so the rest of the code runs.
    Returns (robot_id, num_joints, end_effector_link_index).
    """
    # Franka Panda has 7 arm joints + 2 finger joints. End-effector is often link 11 or 9.
    data_path = pybullet_data.getDataPath()
    # Common paths for Panda in pybullet_data or add-on packages
    candidates = [
        os.path.join(data_path, "franka_panda", "panda.urdf"),
        os.path.join(data_path, "franka_panda", "panda_finger.urdf"),
        "franka_panda/panda.urdf",
    ]
    urdf_path = None
    for c in candidates:
        if os.path.isfile(c):
            urdf_path = c
            break
    if urdf_path is None:
        # Fallback: use Kuka arm (included in pybullet_data) for same FK/IK concepts
        urdf_path = os.path.join(data_path, "kuka_iiwa", "kuka_iiwa.urdf")
    if not os.path.isfile(urdf_path):
        raise FileNotFoundError(
            "Franka/Kuka URDF not found. Install pybullet_data and ensure franka_panda or kuka_iiwa is present."
        )
    # Load robot at default position
    robot_id = p.loadURDF(urdf_path, [0, 0, 0], useFixedBase=True)
    num_joints = p.getNumJoints(robot_id)
    # End-effector: last non-fixed link (Panda: link 11; Kuka: often 6)
    ee_link = num_joints - 1
    for i in range(num_joints - 1, -1, -1):
        #p.getJointInfo(robot_id, i) – Returns a tuple of info for joint index i.
        #info[2] – The joint type (see PyBullet docs). For example:
        #p.JOINT_REVOLUTE – revolute (hinge)
        #p.JOINT_PRISMATIC – prismatic (slider)
        #p.JOINT_FIXED – fixed (no motion)
        #So for each joint we only care whether it’s fixed or not; that’s what we use next.Many URDFs add a fixed joint at the very end (e.g. to attach a tool or frame). That joint doesn’t move; for FK/IK we want the link of the last joint that does move.
        #So we skip fixed joints and treat the last non-fixed one as “the joint whose link is the end-effector.”

        info = p.getJointInfo(robot_id, i)
        if info[2] != p.JOINT_FIXED:
            ee_link = i
            break
    return robot_id, num_joints, ee_link


# Start simulation (DIRECT = no window)
cid = start_pybullet_environment(use_gui=False)
robot_id, num_joints, ee_link = load_franka_panda()
print("✅ PyBullet started. Robot id=%d, joints=%d, end-effector link=%d" % (robot_id, num_joints, ee_link))

✅ PyBullet started. Robot id=0, joints=12, end-effector link=10


In [ ]:
# ============================================================
# Step 4.2: Forward kinematics – joint angles to end-effector pose
# ============================================================
# FK answers: "If the robot has these joint angles, where is the end-effector?"
# PyBullet: set joint positions, then get link state of the end-effector link.

def get_arm_joint_indices(robot_id: int, num_joints: int):
    """Return list of controllable (revolute/prismatic) joint indices for the arm (no fingers).
        We only use revolute and prismatic because those are the only joint types that are controllable — they have a degree of freedom you can actually set. The others either don’t move or aren’t used for “arm” control.
    
    """
    indices = []
    for i in range(num_joints):
        info = p.getJointInfo(robot_id, i)
        jtype = info[2]
        if jtype in (p.JOINT_REVOLUTE, p.JOINT_PRISMATIC):
            indices.append(i)
    return indices


def forward_kinematics(robot_id: int, ee_link: int, joint_positions: List[float]) -> Tuple[Tuple[float, float, float], Tuple[float, float, float, float]]:
    """
    Compute end-effector pose from joint angles.
    joint_positions: list of angles (rad) for each controllable joint.
    Returns (position_xyz, orientation_quat_xyzw).
    """
    num_j = p.getNumJoints(robot_id)
    joint_indices = get_arm_joint_indices(robot_id, num_j)
    for idx, pos in zip(joint_indices, joint_positions):
        #Sets joint idx to angle pos (radians). This updates the simulated robot configuration.
        print(f"idx: {idx}, pos: {pos}")
        p.resetJointState(robot_id, idx, pos)
    #Returns the world-frame pose of the link with index ee_link (the end-effector), given the current joint configuration we just set.
    link_state = p.getLinkState(robot_id, ee_link)
    pos = link_state[0]
    orn = link_state[1]
    return (pos, orn)


# Get joint indices for later use
arm_joint_indices = get_arm_joint_indices(robot_id, num_joints)
# Default pose: all zeros (or a safe home position)
default_joints = [0.0] * len(arm_joint_indices)
print(f"default_joints: {default_joints}")
pos_ee, orn_ee = forward_kinematics(robot_id, ee_link, default_joints)
print("✅ Forward kinematics: default joints -> end-effector position =", [round(x, 3) for x in pos_ee])

default_joints: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
idx: 0, pos: 0.0
idx: 1, pos: 0.0
idx: 2, pos: 0.0
idx: 3, pos: 0.0
idx: 4, pos: 0.0
idx: 5, pos: 0.0
idx: 6, pos: 0.0
idx: 9, pos: 0.0
idx: 10, pos: 0.0
✅ Forward kinematics: default joints -> end-effector position = [0.081, 0.007, 0.848]


In [12]:
# ============================================================
# Step 4.3: Inverse kinematics – target pose to joint angles
# ============================================================
# IK answers: "What joint angles bring the end-effector to this pose?"
# PyBullet's calculateInverseKinematics uses numerical optimization.
# We only care about position (target x,y,z); orientation can be fixed or free.

def inverse_kinematics(
    robot_id: int,
    ee_link: int,
    target_position: Tuple[float, float, float],
    joint_indices: List[int],
    max_iterations: int = 100,
    residual_threshold: float = 1e-5,
) -> Optional[List[float]]:
    """
    Solve IK for target end-effector position. Uses PyBullet's built-in IK.
    Returns list of joint angles (rad) if a solution is found, else None.
    """
    # Use current joint state as seed
    seed = []
    for i in joint_indices:
        seed.append(p.getJointState(robot_id, i)[0])
    # Target orientation: keep current or use identity (pointing down)
    current_orn = p.getLinkState(robot_id, ee_link)[1]
    joint_poses = p.calculateInverseKinematics(
        robot_id,
        ee_link,
        target_position,
        current_orn,
        maxNumIterations=max_iterations,
        residualThreshold=residual_threshold,
    )
    if joint_poses is None:
        return None
    # Return only the arm joints (first len(joint_indices) values)
    return list(joint_poses[: len(joint_indices)])


# Test IK: ask for a point in front of the robot
test_target = (0.3, 0.0, 0.2)
ik_solution = inverse_kinematics(robot_id, ee_link, test_target, arm_joint_indices)
if ik_solution is not None:
    print("✅ Inverse kinematics: target", test_target, "-> solution found, joints:", [round(x, 3) for x in ik_solution])
else:
    print("⚠️ IK: no solution for target", test_target, "(may be out of reach)")

✅ Inverse kinematics: target (0.3, 0.0, 0.2) -> solution found, joints: [-0.41, 2.176, -0.002, 1.187, 0.002, 0.989, -0.408, 0.0, 0.324]


In [13]:
# ============================================================
# Step 4.4: Collision checking and reachability
# ============================================================
# - Reachability: run IK for the target point; if IK finds a solution, the point is reachable.
# - Collision: at the IK solution (or any configuration), check if the robot collides with
#   itself (self-collision) or with other bodies (table, objects). We use getClosestPoints
#   with distance 0 to detect contact.

def robot_self_collision(robot_id: int, joint_indices: List[int], joint_positions: List[float]) -> bool:
    """Set robot to joint_positions and check if any two links are in collision.
    Purpose: Given a set of joint angles, move the robot to that configuration and check if any two of its links are in collision.
    """
    #After this loop, the robot is in the configuration defined by joint_positions.

    for i, idx in enumerate(joint_indices):
        p.resetJointState(robot_id, idx, joint_positions[i])
    # Check all pairs of links (simplified: use getClosestPoints with robot vs itself)
    num_links = p.getNumJoints(robot_id) + 1  # base + links
    for i in range(num_links):
        for j in range(i + 2, num_links):  # skip adjacent
            #For each pair (i, j), p.getClosestPoints(robot_id, robot_id, 0.0, i, j) returns contact points between link i and link j if they are at distance ≤ 0 (i.e. penetrating).
            #If any pair has at least one contact point, we return True (self-collision). If none do, we return False.
            pts = p.getClosestPoints(robot_id, robot_id, 0.0, i, j)
            if pts:
                return True
    return False


def robot_environment_collision(robot_id: int, other_bodies: List[int]) -> bool:
    """Check if robot is in collision with any of the given body IDs."""
    #other_bodies is a list of PyBullet body IDs (e.g. table, walls, objects).
    #Purpose: Check if the robot is in collision with any of the given bodies.
    for other in other_bodies:
        pts = p.getClosestPoints(robot_id, other, 0.0)
        if pts:
            return True
    return False


def is_reachable(
    robot_id: int,
    ee_link: int,
    target_xyz: Tuple[float, float, float],
    joint_indices: List[int],
    obstacle_ids: Optional[List[int]] = None,
) -> Tuple[bool, Optional[List[float]], str]:
    """
    Check if target_xyz is reachable: solve IK and optionally check collisions.
    Returns (reachable, solution_joints, message).
    """
    sol = inverse_kinematics(robot_id, ee_link, target_xyz, joint_indices)
    if sol is None:
        return False, None, "IK failed: target out of workspace or unreachable"
    if robot_self_collision(robot_id, joint_indices, sol):
        return False, None, "Self-collision at IK solution"
    if obstacle_ids and robot_environment_collision(robot_id, obstacle_ids):
        return False, None, "Robot would collide with environment at IK solution"
    return True, sol, "Reachable"

---
# Part 5: Feasibility Checking – Putting It Together

We now implement a **feasibility checker** that, given:
- the **symbolic plan** (list of `RobotAction`),
- **grounded detections** (objects with `position_3d`),
- the **PyBullet** robot and (optionally) table/obstacles,

does the following for each action:

1. **Reachability**: For pick/move_to, the target object’s 3D position must be reachable (IK succeeds, no self-collision, no collision with environment). For place, the placement location (e.g. “near laptop”) must be reachable.
2. **Collision**: Any configuration used (e.g. IK solution for pick or place) must not be in collision with the environment or itself.
3. **Stability**: For place actions, we require that the placement is on the table (z ≈ table height). You can extend this to “object must be supported by another object” later.

The checker returns **feasible** (True/False) and a short **reason** string.

In [14]:
# ============================================================
# FeasibilityChecker: reachability, collision, stability
# ============================================================

@dataclass
class GroundedPlan:
    """Plan with each action's target/location bound to 3D positions."""
    actions: List[RobotAction]
    # Maps action index -> target object -> DetectedObject (with position_3d)
    target_objects: Dict[int, Optional[DetectedObject]]
    # Maps action index -> location object (for place/move_to)
    location_objects: Dict[int, Optional[DetectedObject]]


def build_grounded_plan(
    actions: List[RobotAction],
    detections: List[DetectedObject],
) -> GroundedPlan:
    """Bind each action's target and location to detections (with position_3d)."""
    target_objects = {}
    location_objects = {}
    for i, a in enumerate(actions):
        target_objects[i] = match_symbol_to_detection(a.target, detections)
        if a.location:
            location_objects[i] = match_symbol_to_detection(a.location, detections)
        else:
            location_objects[i] = None
    return GroundedPlan(actions=actions, target_objects=target_objects, location_objects=location_objects)


def check_stability_place(
    place_position_3d: Tuple[float, float, float],
    table_z: float = TABLE_Z,
    tolerance: float = 0.05,
) -> Tuple[bool, str]:
    """Simple stability: placement should be on table (z ≈ table_z)."""
    if abs(place_position_3d[2] - table_z) <= tolerance:
        return True, "On table"
    return False, "Placement not on table (z=%.2f, table=%.2f)" % (place_position_3d[2], table_z)

In [15]:
def compute_place_position(
    relation: Optional[str],
    location_obj: Optional[DetectedObject],
    table_z: float = TABLE_Z,
) -> Optional[Tuple[float, float, float]]:
    """
    Compute 3D position for a place action: "near X" -> next to location_obj, on table.
    "on X" -> on top of location (we approximate as slightly above its center).
    """
    if location_obj is None or location_obj.position_3d is None:
        return None
    x, y, z = location_obj.position_3d
    if relation == "on":
        # Place on top: raise z slightly (object height not modeled here)
        return (x, y, z + 0.05)
    # "near" or default: same height as table, offset slightly in x,y
    return (x + 0.05, y, table_z)


def check_plan_feasibility(
    grounded: GroundedPlan,
    robot_id: int,
    ee_link: int,
    joint_indices: List[int],
    obstacle_body_ids: Optional[List[int]] = None,
) -> Tuple[bool, List[str]]:
    """
    Check if the grounded plan is feasible: reachability, collision, stability.
    Returns (is_feasible, list of reason strings).
    """
    reasons = []
    obstacle_ids = obstacle_body_ids or []

    for i, action in enumerate(grounded.actions):
        a = action.action
        target_obj = grounded.target_objects.get(i)
        location_obj = grounded.location_objects.get(i)

        if a == ActionType.PICK or a == ActionType.GRASP:
            if target_obj is None or target_obj.position_3d is None:
                reasons.append("Pick target '%s' not grounded (no detection)" % action.target)
                return False, reasons
            ok, sol, msg = is_reachable(robot_id, ee_link, target_obj.position_3d, joint_indices, obstacle_ids)
            if not ok:
                reasons.append("Pick '%s': %s" % (action.target, msg))
                return False, reasons
            reasons.append("Pick '%s': reachable" % action.target)

        elif a == ActionType.PLACE:
            if target_obj is None:
                reasons.append("Place target '%s' not grounded" % action.target)
                return False, reasons
            place_pos = compute_place_position(action.relation, location_obj)
            if place_pos is None:
                reasons.append("Place location '%s' not grounded" % (action.location or ""))
                return False, reasons
            stable, stab_msg = check_stability_place(place_pos)
            if not stable:
                reasons.append("Place: %s" % stab_msg)
                return False, reasons
            ok, sol, msg = is_reachable(robot_id, ee_link, place_pos, joint_indices, obstacle_ids)
            if not ok:
                reasons.append("Place at '%s': %s" % (action.location, msg))
                return False, reasons
            reasons.append("Place at '%s': reachable and stable" % action.location)

        elif a == ActionType.MOVE_TO:
            loc = location_obj or target_obj
            if loc is None or loc.position_3d is None:
                reasons.append("Move_to target '%s' not grounded" % (action.target or action.location))
                return False, reasons
            ok, sol, msg = is_reachable(robot_id, ee_link, loc.position_3d, joint_indices, obstacle_ids)
            if not ok:
                reasons.append("Move_to '%s': %s" % (action.target, msg))
                return False, reasons
            reasons.append("Move_to '%s': reachable" % action.target)

    return True, reasons

---
# Part 6: Full Pipeline – Command to Feasibility

We now chain everything:

1. **Natural language command** → `parse_command_with_nim(command)` → **symbolic plan** (list of `RobotAction`).
2. **Scene image** → `run_object_detection(image_path)` → **detections**; then `ground_detections_to_world(detections)` to set 3D positions.
3. **Grounded plan** → `build_grounded_plan(actions, detections)`.
4. **Feasibility** → `check_plan_feasibility(grounded, robot_id, ee_link, joint_indices, obstacle_ids)`.

Optionally add a **table** in PyBullet so the robot must not collide with it; then pass its body ID as an obstacle.

In [16]:
# ============================================================
# Optional: create a table in PyBullet for collision checking
# ============================================================
# So the robot must not go through the table. We use a simple box as table.

def create_table():
    """Create a box as table surface, return body ID. Table top at z=TABLE_Z (0).
    Needed so that the robot has a surface to pick from and place to 
    """
    #This does not create a visible or simulated “object” yet. It only defines a collision shape: the geometry used for collisions.
    #p.GEOM_BOX – “Use a box shape.”
    table_half_extents = [0.5, 0.5, 0.02]
    col_id = p.createCollisionShape(p.GEOM_BOX, halfExtents=table_half_extents)
    
    #The function below creates the actual rigid body in the world by turning that shape into an object with a position.
    #First 0 – Mass. 0 = static (fixed in place, doesn’t move when hit). So the table doesn’t slide or tip.
    #col_id – The collision shape from step 1 (the thin box).
    #-1 – Visual shape ID. -1 means “no separate visual mesh”; PyBullet will draw the collision shape.
    #[0, 0, -0.02] – Position of the body’s center in the world: (x, y, z) = (0, 0, -0.02).
    #Why z = -0.02?
    #The box is 0.04 m thick (half extent 0.02 in z). If the center is at z = -0.02:
    #Bottom of the box: z = -0.02 - 0.02 = -0.04
    #Top of the box: z = -0.02 + 0.02 = 0
    #So the top of the table is at z = 0. The comment says “Table top at z=TABLE_Z (0)” — that’s exactly what this position achieves.
    body = p.createMultiBody(0, col_id, -1, [0, 0, -0.02])
    return body


# Create table and keep list of obstacles for feasibility
table_id = create_table()
obstacle_ids = [table_id]
print("✅ Table created (body id=%d). Obstacles: %s" % (table_id, obstacle_ids))

✅ Table created (body id=1). Obstacles: [1]


In [17]:
# ============================================================
# Full pipeline: natural language -> feasibility result
# ============================================================

def run_full_pipeline(
    command: str,
    image_path: str,
    robot_id: int,
    ee_link: int,
    joint_indices: List[int],
    obstacle_ids: Optional[List[int]] = None,
    nim_client: Optional[OpenAI] = None,
):
    """
    1. Parse command with NIM -> symbolic plan
    2. Run YOLO on image -> detections, ground to 3D
    3. Build grounded plan
    4. Check feasibility
    Returns (is_feasible, plan, grounded_plan, reasons).

    Where to get the robot/simulation parameters (run Part 4 & optional Part 6 first):
    - robot_id, ee_link: from load_franka_panda() in Part 4 (Cell 15).
    - joint_indices: from get_arm_joint_indices(robot_id, num_joints) in Part 4 (Cell 16).
    - obstacle_ids: optional list of PyBullet body IDs to avoid (e.g. table from create_table() in Part 6, Cell 23). Use [] or None if no obstacles.
    """
    # Step 1: Natural language -> symbolic plan
    plan = parse_command_with_nim(command, client=nim_client)
    print("Step 1 – Symbolic plan:", [a.to_tuple() for a in plan])

    # Step 2: Object detection and grounding
    detections = run_object_detection(image_path)
    ground_detections_to_world(detections)
    print("Step 2 – Detected %d objects (with 3D positions)" % len(detections))

    # Step 3: Grounded plan
    grounded = build_grounded_plan(plan, detections)
    print("Step 3 – Grounded plan built")

    # Step 4: Feasibility
    feasible, reasons = check_plan_feasibility(
        grounded, robot_id, ee_link, joint_indices, obstacle_ids
    )
    print("Step 4 – Feasibility:", "YES" if feasible else "NO")
    for r in reasons:
        print("   ", r)
    return feasible, plan, grounded, reasons


# Run on a sample command and scene (use scene_1.jpg if available)
command_demo = "Pick up the cup and place it near the laptop"
image_demo = "scene_1.jpg"
if not os.path.isfile(image_demo):
    image_demo = "scene.jpg"
if os.path.isfile(image_demo):
    feasible, plan, grounded, reasons = run_full_pipeline(
        command_demo, image_demo, robot_id, ee_link, arm_joint_indices, obstacle_ids
    )
    print("\n🎯 Result: Command is", "FEASIBLE" if feasible else "NOT FEASIBLE")
else:
    print("No scene image (scene_1.jpg or scene.jpg) found. Run run_full_pipeline with your own image path.")

Step 1 – Symbolic plan: [('pick', 'cup', None, None), ('place', 'cup', 'laptop', 'near')]
Step 2 – Detected 0 objects (with 3D positions)
Step 3 – Grounded plan built
Step 4 – Feasibility: NO
    Pick target 'cup' not grounded (no detection)

🎯 Result: Command is NOT FEASIBLE


---
# Summary and Next Steps

## What we built

1. **Natural language → symbolic plan**: NIM LLM (or fallback parser) turns a command into a list of `RobotAction` (pick, place, move_to, etc.).
2. **Grounding**: YOLO detects objects in the scene; we assign 3D positions on a table plane and match plan symbols to detections.
3. **PyBullet + Franka**: We load the robot, compute **forward kinematics** (joints → end-effector pose) and **inverse kinematics** (target pose → joint angles).
4. **Feasibility**: For each action we check **reachability** (IK + no collision), **collision** (robot vs. self and environment), and **stability** (place on table).

## Try it yourself

- Change `command_demo` and re-run the pipeline.
- Use different scene images and ensure object names in the command match YOLO labels (e.g. cup, laptop, book).
- Start NIM with `docker-compose -f docker-compose-nim.yml up -d` to use the real LLM parser.

## Extensions

- **Multiple bindings**: If several detections match "mug", try each and pick the first feasible one.
- **Trajectory**: Use IK waypoints to plan a collision-free path, not just the goal pose.
- **Real robot**: Replace PyBullet with your robot’s FK/IK and collision checker; keep the same pipeline (plan → ground → feasibility).